In [1]:
!pip install --upgrade --quiet google-cloud-vectorsearch pandas requests google-adk

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastmcp 2.13.3 requires mcp!=1.21.1,<1.23,>=1.19.0, but you have mcp 1.26.0 which is incompatible.
streamlit 1.50.0 requires pandas<3,>=1.4.0, but you have pandas 3.0.1 which is incompatible.


In [3]:
from google.cloud import bigquery

client = bigquery.Client()


In [ ]:
import os
from google.cloud import vectorsearch_v1beta


PROJECT_ID = "project-843a9db4-70a9-4565-b6c"
LOCATION = "us-central1"
COLLECTION_ID = "london-travel-agent-demo"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

admin_client = vectorsearch_v1beta.VectorSearchServiceClient()
data_client = vectorsearch_v1beta.DataObjectServiceClient()
search_client = vectorsearch_v1beta.DataObjectSearchServiceClient()

parent = f"projects/{PROJECT_ID}/locations/{LOCATION}"
collection_path = f"{parent}/collections/{COLLECTION_ID}"

print(f"Project: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Collection: {COLLECTION_ID}")

Project: project-843a9db4-70a9-4565-b6c
Location: us-central1
Collection: london-travel-agent-demo


In [12]:
import io
import pandas as pd
import requests
import warnings
warnings.filterwarnings('ignore')


# Source: Inside Airbnb (London, Sept 2025)
DATA_URL = "https://data.insideairbnb.com/united-kingdom/england/london/2025-09-14/data/listings.csv.gz"

print('Downloading data...')

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}
response= requests.get(DATA_URL, headers= headers)
response.raise_for_status()

df= pd.read_csv(io.BytesIO(response.content), compression="gzip")
df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,13913,https://www.airbnb.com/rooms/13913,20250914034649,2025-09-16,city scrape,Holiday London DB Room Let-on going,My bright double bedroom with a large window h...,Finsbury Park is a friendly melting pot commun...,https://a0.muscache.com/pictures/miso/Hosting-...,54730,...,4.87,4.78,4.78,NaN,f,2,1,1,0,0.30
1,15400,https://www.airbnb.com/rooms/15400,20250914034649,2025-09-16,city scrape,Bright Chelsea Apartment. Chelsea!,Lots of windows and light. St Luke's Gardens ...,It is Chelsea.,https://a0.muscache.com/pictures/428392/462d26...,60302,...,4.84,4.93,4.74,NaN,f,1,1,0,0,0.51
2,17402,https://www.airbnb.com/rooms/17402,20250914034649,2025-09-16,city scrape,Very Central Modern 3-Bed/2 Bath By Oxford St W1,"You'll have a great time in this beautiful, cl...","Fitzrovia is a very desirable trendy, arty and...",https://a0.muscache.com/pictures/39d5309d-fba7...,67564,...,4.72,4.89,4.61,NaN,f,2,2,0,0,0.32
3,24328,https://www.airbnb.com/rooms/24328,20250914034649,2025-09-18,previous scrape,Battersea live/work artist house,"Artist house by SW Battersea Park, bright high...","- Battersea is a quiet family area, easy acces...",https://a0.muscache.com/pictures/9194b40f-c627...,41759,...,4.93,4.60,4.65,NaN,f,1,1,0,0,0.53
4,36274,https://www.airbnb.com/rooms/36274,20250914034649,2025-09-15,city scrape,Bright 1 bedroom apt off brick lane in Shoreditch,*Update June '25- Pump Installed to improve wa...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,133271,...,4.46,4.85,4.54,NaN,t,2,2,0,0,0.09


# Data Cleaning


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 96871 entries, 0 to 96870
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            96871 non-null  int64  
 1   listing_url                                   96871 non-null  str    
 2   scrape_id                                     96871 non-null  int64  
 3   last_scraped                                  96871 non-null  str    
 4   source                                        96871 non-null  str    
 5   name                                          96871 non-null  str    
 6   description                                   94421 non-null  str    
 7   neighborhood_overview                         41208 non-null  str    
 8   picture_url                                   96865 non-null  str    
 9   host_id                                       96871 non-null  int64  
 1

In [14]:
df['price']

0         $70.00
1        $149.00
2        $411.00
3            NaN
4        $210.00
          ...   
96866    $298.00
96867     $66.00
96868    $350.00
96869     $40.00
96870    $139.00
Name: price, Length: 96871, dtype: str

In [16]:
cols = [
    "id",
    "name",
    "description",
    "price",
    "neighborhood_overview",
    "listing_url",
    "instant_bookable",
    "neighbourhood_cleansed",
]
df = df[cols].copy()

# clean price column remove $ and convert to float
df['price']= df['price'].astype(str).str.replace("[\$,]", "", regex= True).astype(float)


# Fill NaNs in text fields (Critical to avoid API errors)
str_cols = [
    "name",
    "neighbourhood_cleansed",
    "instant_bookable",
    "listing_url",
    "description",
    "neighborhood_overview",
]
for col in str_cols:
    df[col] = df[col].fillna("").astype(str)

# Normalize boolean string
df["instant_bookable"] = df["instant_bookable"].str.lower()

# Subset for demo speed
df_demo = df.head(2000).reset_index(drop=True)
print(f"Loaded & Cleaned {len(df_demo)} listings.")






Loaded & Cleaned 2000 listings.



# Part 3: Create Collection
## Create a Vector Search 2.0 Collection
A Collection is a schema-enforced container for your data in Vector Search 2.0. Think of it as a table in a traditional database, but optimized for vector operations.

## Collection Schemas
Each Collection has two schemas:

*Data Schema*: Defines the structure of your data fields using JSON Schema format. All Data Objects must conform to this schema.

*Vector Schema*: Defines your embedding fields with their dimensions and configurations. You can have multiple vector fields per object (e.g., text_embedding, image_embedding).

## Auto-Embeddings Feature
One of Vector Search 2.0's most powerful features is automatic embedding generation. When you configure vertex_embedding_config in your vector schema, the service automatically generates embeddings using Vertex AI models. This means you don't need to:

Manage embedding model infrastructure
Pre-compute embeddings before ingestion
Handle embedding API calls yourself

In [19]:

collection_config = {
    # DATA SCHEMA: Defines the structure of your data fields
    # All fields in Data Objects must match these types
    "data_schema": {
        "type": "object",
        "properties": {
            "name": {"type": "string"},  # Listing title
            "price": {"type": "number"},  # Price per night (GBP)
            "neighborhood": {"type": "string"},  # London neighborhood
            "listing_url": {"type": "string"},  # Airbnb URL
            "instant_bookable": {"type": "string"},  # "t" or "f"
            "description": {"type": "string"},  # Full listing description
            "neighborhood_overview": {"type": "string"},  # Area description
        },
    },
    # VECTOR SCHEMA: Defines embedding fields and their configurations
    "vector_schema": {
        "description_embedding": {
            "dense_vector": {
                # Embedding dimensions (768 for gemini-embedding-001)
                "dimensions": 768,
                # AUTO-EMBEDDING CONFIGURATION
                # Vector Search 2.0 will automatically generate embeddings
                # using the specified Vertex AI model
                "vertex_embedding_config": {
                    "model_id": "gemini-embedding-001",
                    # text_template: Combines multiple fields into embedding input
                    # This creates richer semantic embeddings by including both
                    # the description AND neighborhood context
                    "text_template": "Description: {description}. Neighborhood: {neighborhood_overview}.",
                    # task_type: Optimizes embeddings for retrieval use cases
                    # Use RETRIEVAL_DOCUMENT for documents being indexed
                    "task_type": "RETRIEVAL_DOCUMENT",
                },
            }
        }
    },
}

# Create the Collection (or skip if it already exists)
try:
    existing = admin_client.get_collection(name=collection_path)
    print(f"Collection '{COLLECTION_ID}' already exists.")
except Exception:
    print(f"Creating Collection '{COLLECTION_ID}'...")
    request = vectorsearch_v1beta.CreateCollectionRequest(
        parent=parent, collection_id=COLLECTION_ID, collection=collection_config
    )
    operation = admin_client.create_collection(request=request)
    operation.result()  # Wait for completion
    print(f"Collection '{COLLECTION_ID}' created successfully!")

Creating Collection 'london-travel-agent-demo'...
Collection 'london-travel-agent-demo' created successfully!
